In [1]:
!pip install confluent-kafka


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 39.4 MB/s eta 0:00:00


In [ ]:
BOOTSTRAP_SERVERS = ""   
API_KEY = ""
API_SECRET = ""
TOPIC = "incidents_stream"

config = {
    "bootstrap.servers": BOOTSTRAP_SERVERS,
    "security.protocol": "SASL_SSL",
    "sasl.mechanism": "PLAIN",
    "sasl.username": API_KEY,
    "sasl.password": API_SECRET,
}


In [3]:
from confluent_kafka import Producer
import json
import random
import time
from datetime import datetime

producer = Producer(config)

alert_codes = ["FLOOD", "FIRE", "ACCIDENT", "PROTEST", "OTHER"]
tags = ["critical", "warning", "info"]

# coordonate aproximative (București)
lat_min, lat_max = 44.38, 44.48
lon_min, lon_max = 26.05, 26.20

hotspots = [
    (44.4325, 26.1039),
    (44.4459, 26.0979),
    (44.4268, 26.1025),
    (44.4781, 26.0556),
    (44.4050, 26.1000),
]


In [4]:
def generate_incident():
    alert = random.choice(alert_codes)
    descriptions = {
        "FLOOD": "Inundație în zonă rezidențială.",
        "FIRE": "Incendiu la o clădire.",
        "ACCIDENT": "Accident rutier la intersecție.",
        "PROTEST": "Protest în centrul orașului.",
        "OTHER": "Incident neclasificat raportat.",
    }

    if random.random() < 0.85:
        base_lat, base_lon = random.choice(hotspots)
        lat = base_lat + random.uniform(-0.006, 0.006)
        lon = base_lon + random.uniform(-0.010, 0.010)
    else:
        lat = random.uniform(lat_min, lat_max)
        lon = random.uniform(lon_min, lon_max)

    return {
        "id": random.randint(1, 1_000_000),
        "reported_at": datetime.utcnow().isoformat(),
        "lat": round(lat, 6),
        "lon": round(lon, 6),
        "alert_code": alert,
        "description": descriptions[alert],
        "tag": random.choice(tags),
    }

In [10]:
print("Starting producer... (CTRL+STOP to halt)")

try:
    while True:
        incident = generate_incident()
        producer.produce(
            TOPIC,
            value=json.dumps(incident).encode("utf-8")
        )
        producer.poll(0)
        print("Sent:", incident)
        time.sleep(1)
except KeyboardInterrupt:
    print("Producer stopped.")
finally:
    producer.flush()


Starting producer... (CTRL+STOP to halt)
Sent: {'id': 503832, 'reported_at': '2026-01-19T21:39:44.861237', 'lat': 44.43092, 'lon': 26.109672, 'alert_code': 'FLOOD', 'description': 'Inundație în zonă rezidențială.', 'tag': 'critical'}


/tmp/ipython-input-1382720143.py:21: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "reported_at": datetime.utcnow().isoformat(),


Sent: {'id': 994244, 'reported_at': '2026-01-19T21:39:45.861601', 'lat': 44.449669, 'lon': 26.103625, 'alert_code': 'OTHER', 'description': 'Incident neclasificat raportat.', 'tag': 'info'}
Sent: {'id': 392189, 'reported_at': '2026-01-19T21:39:46.862042', 'lat': 44.434838, 'lon': 26.099022, 'alert_code': 'FIRE', 'description': 'Incendiu la o clădire.', 'tag': 'warning'}
Sent: {'id': 583283, 'reported_at': '2026-01-19T21:39:47.862587', 'lat': 44.426858, 'lon': 26.09574, 'alert_code': 'FLOOD', 'description': 'Inundație în zonă rezidențială.', 'tag': 'info'}
Sent: {'id': 149781, 'reported_at': '2026-01-19T21:39:48.863125', 'lat': 44.410394, 'lon': 26.109937, 'alert_code': 'PROTEST', 'description': 'Protest în centrul orașului.', 'tag': 'warning'}
Producer stopped.
